#load the data

In [ ]:
import kagglehub
import pandas as pd
import os
# Download latest version
path = kagglehub.dataset_download("datafiniti/consumer-reviews-of-amazon-products")

print("Path to dataset files:", path)

# List all files in the directory
for file in os.listdir(path):
    print(file)
df1 = pd.read_csv(os.path.join(path, "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv"))
df2 = pd.read_csv(os.path.join(path, "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"))


In [ ]:
df2['primaryCategories'].unique()

array(['Health & Beauty', 'Electronics', 'Office Supplies',
       'Animals & Pet Supplies', 'Home & Garden', 'Electronics,Furniture',
       'Toys & Games,Electronics', 'Electronics,Media',
       'Office Supplies,Electronics'], dtype=object)

#merge and clean the dataset

In [ ]:

#  Load the original dataset
df_original = pd.concat([df1, df2], ignore_index=True)

# Load balanced and augmented dataset
balanced_df = pd.read_csv("balanced_reviews.csv")

#  Make sure column names match for merging
balanced_df.rename(columns={"text": "reviews.text"}, inplace=True)

merged_df = balanced_df.merge(
    df_original[["reviews.text", "name", "reviews.rating", "primaryCategories"]],
    on="reviews.text",
    how="left"  # Keeps all balanced reviews
)

#  renaming for simplicity
merged_df.rename(columns={"reviews.text": "review", "reviews.rating": "rating"}, inplace=True)

#Drop rows with missing values
merged_df.dropna(subset=["review", "name", "rating"], inplace=True)

# Convert rating to integer, if needed
merged_df["rating"] = merged_df["rating"].astype(int)

# Keep only the relevant columns
final_df = merged_df[["review", "name", "rating", "label","primaryCategories"]].copy()




In [ ]:
final_df.head()

,review,name,rating,label,primaryCategories
1,I like the price but haven't had them long eno...,AmazonBasics AA Performance Alkaline Batteries...,5,2,Health & Beauty
2,Just what I wanted.,AmazonBasics AAA Performance Alkaline Batterie...,5,2,Health & Beauty
3,Just what I wanted.,AmazonBasics AA Performance Alkaline Batteries...,5,2,Health & Beauty
6,Nice quality and friendly packaging... besides...,AmazonBasics AAA Performance Alkaline Batterie...,5,2,Health & Beauty
7,Nice quality and friendly packaging... besides...,AmazonBasics AA Performance Alkaline Batteries...,5,2,Health & Beauty


#grouping and aggregation

In [ ]:
# Group by primaryCategories
for category, group in final_df.groupby("primaryCategories"):
    print(f"--- {category} ---")

    # Group and sort products by average rating and review count
    product_stats = (
        group.groupby("name")
        .agg(avg_rating=("rating", "mean"), review_count=("review", "count"))
        .sort_values(by=["avg_rating", "review_count"], ascending=False)
    )

    # Top 3 products by rating and number of reviews
    top3 = product_stats.head(3)

    # Worst product based on the lowest rating
    worst = product_stats.tail(1)

    # Sample complaints from negative reviews
    complaints = group[group['label'] == 0]["review"].sample(min(5, len(group[group['label'] == 0]))).tolist()

    # Sample praises from positive reviews
    praises = group[group['label'] == 2]["review"].sample(min(5, len(group[group['label'] == 2]))).tolist()

    # Now, create the summary input-output pair as needed
    # For example:
    print("Top 3 Products:")
    print(top3[['avg_rating', 'review_count']])

    print("\nWorst Product:")
    print(worst[['avg_rating', 'review_count']])

    print("\nComplaints:")
    print(complaints)

    print("\nPraises:")
    print(praises)

    print("\n--- End of Category ---")



--- Animals & Pet Supplies ---
Top 3 Products:
                                                    avg_rating  review_count
name                                                                        
AmazonBasics Double-Door Folding Metal Dog Crat...         3.0             1

Worst Product:
                                                    avg_rating  review_count
name                                                                        
AmazonBasics Double-Door Folding Metal Dog Crat...         3.0             1

Complaints:
[]

Praises:
[]

--- End of Category ---
--- Electronics ---
Top 3 Products:
                                                    avg_rating  review_count
name                                                                        
Certified Refurbished Amazon Echo                          5.0             4
AmazonBasics 16-Gauge Speaker Wire - 100 Feet              5.0             2
Amazon Echo (2nd Generation) Smart Assistant Oa...         5.0             1


# using generative AI model

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Initialize client
api_key = userdata.get('OPENAI_API')
client = OpenAI(api_key=api_key)

# Summary generation function
def generate_summary(category, top_products, worst_product, complaints, praises):
    prompt = f"""
Write a blog-style summary for the following product category. Use a clear, engaging tone with headings for each section.

Category: {category}

Top 3 Products:
{top_products}

Worst Product:
{worst_product}

Common Complaints:
{complaints}

Customer Praises:
{praises}
"""

    # Send prompt to GPT-4
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant who writes clean and engaging product summaries."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=600
    )

    return response.choices[0].message.content.strip()

# Example data
category = "Electronics"
top_products = """
1. Certified Refurbished Amazon Echo - avg_rating: 5.0, review_count: 4, Summary: A reliable and highly rated smart speaker.
2. AmazonBasics 16-Gauge Speaker Wire - 100 Feet - avg_rating: 5.0, review_count: 2, Summary: A top choice for speaker wiring.
3. Amazon Echo (2nd Generation) Smart Assistant - avg_rating: 5.0, review_count: 1, Summary: The original Echo model that continues to impress.
"""
worst_product = "Oem Amazon Kindle Power Usb Adapter Wall Travel Charger - avg_rating: 1.0, review_count: 4, Complaints: Customers report issues with functionality."
complaints = """
1. Touch screen is poor quality.
2. Stops working after a few months.
3. Screen brightness is too low.
"""
praises = """
1. Easy to set up and use.
2. Reliable for kids' use.
3. Excellent battery life.
"""

# Generate the summary
summary = generate_summary(category, top_products, worst_product, complaints, praises)
print(summary)




# Elevate Your Electronics Game with these Top Picks

With the vast marketplace of electronic products available today, it can be overwhelming to sift through the options and find the best fit for your needs. Today, we're going to save you some time and effort by delving into the top rated electronics on Amazon. 

## Top 3 Products 

### 1. Certified Refurbished Amazon Echo

With an exemplary 5.0 average rating and four glowing reviews, the Certified Refurbished Amazon Echo stands tall in our lineup. This smart speaker is not just reliable, but highly rated as well. It's refurbished to work and look like new, providing excellent value for money.

### 2. AmazonBasics 16-Gauge Speaker Wire - 100 Feet

This top choice for speaker wiring also boasts a perfect 5.0 average rating. With two reviews praising its quality and functionality, the AmazonBasics 16-Gauge Speaker Wire is a worthy addition to your speaker system.

### 3. Amazon Echo (2nd Generation) Smart Assistant

Last but not least,